fine-tune PhoBERT NER — dinh dưỡng tiếng Việt

labels: O / B-FOOD / B-DISEASE / B-NUTRIENT / B-SYMPTOM  
runtime: Colab GPU T4

In [1]:
!pip install -q transformers datasets seqeval underthesea

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.1 MB/s eta 0:00:00


In [7]:
!pip install -q transformers datasets seqeval evaluate underthesea


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


## upload data
chạy cell dưới, chọn file ner_data_augmented.csv

In [2]:
from google.colab import files
uploaded = files.upload()  # chọn ner_data_augmented.csv

Saving ner_data_augmented.csv to ner_data_augmented.csv


## load & split (80/10/10)

In [ ]:
import csv, random

LABEL2ID = {"O": 0, "B-FOOD": 1, "B-DISEASE": 2, "B-NUTRIENT": 3, "B-SYMPTOM": 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

samples = []
with open("ner_data_augmented.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        tokens = row["tokens"].split()
        labels = row["labels"].split()
        if len(tokens) == len(labels):  # skip misaligned
            samples.append({"tokens": tokens, "labels": [LABEL2ID[l] for l in labels]})

random.seed(42)
random.shuffle(samples)

n = len(samples)
train = samples[:int(n*0.8)]
val   = samples[int(n*0.8):int(n*0.9)]
test  = samples[int(n*0.9):]

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

## tokenize + align labels

PhoBERT WordPiece tách 1 word thành nhiều subtoken — subtoken đầu giữ label gốc, còn lại -100 (ignore in loss).

In [ ]:
from transformers import AutoTokenizer

# slow tokenizer — no word_ids(), manual alignment
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

def tokenize_and_align(sample):
    encoding = tokenizer(
        sample["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128,
    )

    # subtoken count per word
    word_subtoken_counts = [
        len(tokenizer.tokenize(w)) for w in sample["tokens"]
    ]

    input_len = len(encoding["input_ids"])
    aligned = [-100]  # [CLS]

    subtoken_idx = 0
    for word_idx, count in enumerate(word_subtoken_counts):
        for i in range(count):
            if subtoken_idx >= input_len - 2:  # max_length guard
                break
            # first subtoken → label, rest → -100
            aligned.append(sample["labels"][word_idx] if i == 0 else -100)
            subtoken_idx += 1

    aligned.append(-100)  # [SEP]

    # pad/trim to match input_ids
    aligned = aligned[:input_len]
    while len(aligned) < input_len:
        aligned.append(-100)

    encoding["labels"] = aligned
    return encoding

from datasets import Dataset

ds_train = Dataset.from_list(train).map(tokenize_and_align, remove_columns=["tokens", "labels"])
ds_val   = Dataset.from_list(val).map(tokenize_and_align,   remove_columns=["tokens", "labels"])
ds_test  = Dataset.from_list(test).map(tokenize_and_align,  remove_columns=["tokens", "labels"])

print("done.")

## load model

In [5]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "vinai/phobert-base",
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_preds  = [[ID2LABEL[p] for p, l in zip(pred, label) if l != -100] for pred, label in zip(preds, labels)]
    true_labels = [[ID2LABEL[l] for p, l in zip(pred, label) if l != -100] for pred, label in zip(preds, labels)]
    result = seqeval.compute(predictions=true_preds, references=true_labels)
    return {"f1": result["overall_f1"], "precision": result["overall_precision"], "recall": result["overall_recall"]}

args = TrainingArguments(
    output_dir="phobert-ner",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="best",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    processing_class=tokenizer,  # new API (was tokenizer=)
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## đánh giá test set

In [10]:
trainer.evaluate(ds_test)

{'eval_loss': 0.003276919247582555,
 'eval_f1': 1.0,
 'eval_precision': 1.0,
 'eval_recall': 1.0,
 'eval_runtime': 0.1484,
 'eval_samples_per_second': 936.797,
 'eval_steps_per_second': 60.656,
 'epoch': 10.0}

lưu model rồi download zip về máy

In [ ]:
trainer.save_model("phobert-ner-final")
tokenizer.save_pretrained("phobert-ner-final")

# zip + download
!zip -r phobert-ner-final.zip phobert-ner-final/
files.download("phobert-ner-final.zip")